In [6]:
#!/usr/bin/env python3
"""
Reading Group Availability Timetable Generator
Creates an interactive HTML timetable from CSV availability data
"""

import csv
import json
from collections import defaultdict
import re

def read_csv_file(filepath):
    """Read CSV file and return parsed data"""
    participants_data = {}
    time_slots = []
    
    with open(filepath, 'r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        # Strip whitespace from headers
        reader.fieldnames = [h.strip() if h else h for h in reader.fieldnames]
        headers = reader.fieldnames
        
        # Get time slot headers (those containing 'Aug 2025')
        time_slots = [h for h in headers if h and 'Aug 2025' in h]
        
        # Read participant data
        for row in reader:
            # Get name, handling both 'Name' field and potential whitespace
            name = None
            for key in ['Name', ' Name', 'Name ']:
                if key in row and row[key]:
                    name = row[key].strip()
                    break
            
            if name:
                participants_data[name] = {}
                for slot in time_slots:
                    # Handle potential whitespace in column names
                    value = row.get(slot, row.get(slot.strip(), 'No'))
                    participants_data[name][slot] = value.strip() if value else 'No'
    
    return participants_data, time_slots

def parse_time_slot(time_str):
    """Parse time slot string and return components for sorting"""
    # Try 24-hour format first: Day DD Aug 2025 HH:00 - HH:00
    pattern_24h = r'(\w+)\s+(\d+)\s+Aug\s+2025\s+(\d{1,2}):00\s*-\s*(\d{1,2}):00'
    match = re.match(pattern_24h, time_str.strip())
    
    if match:
        day_name = match.group(1)
        date = int(match.group(2))
        start_hour = int(match.group(3))  # Already in 24-hour format
        
        return {
            'day_name': day_name,
            'date': date,
            'start_hour': start_hour,  # 0-23 for chronological sorting
            'original': time_str
        }
    
    # Try 12-hour format: Day DD Aug 2025 H:00 AM/PM - H:00 AM/PM
    pattern_12h = r'(\w+)\s+(\d+)\s+Aug\s+2025\s+(\d{1,2}):00\s+(AM|PM)\s*-\s*(\d{1,2}):00\s+(AM|PM)'
    match = re.match(pattern_12h, time_str.strip())
    
    if match:
        day_name = match.group(1)
        date = int(match.group(2))
        start_hour = int(match.group(3))
        start_period = match.group(4)
        
        # Convert to 24-hour format for sorting
        if start_period == 'AM':
            if start_hour == 12:
                hour_24 = 0  # 12 AM = midnight
            else:
                hour_24 = start_hour  # 1 AM = 1, ..., 11 AM = 11
        else:  # PM
            if start_hour == 12:
                hour_24 = 12  # 12 PM = noon
            else:
                hour_24 = start_hour + 12  # 1 PM = 13, ..., 11 PM = 23
        
        return {
            'day_name': day_name,
            'date': date,
            'start_hour': hour_24,
            'original': time_str
        }
    
    print(f"Warning: Could not parse time slot: {time_str}")
    return None

def process_availability_data(participants_data, time_slots):
    """Process availability data and organize by day"""
    availability_matrix = {}
    slots_by_day = defaultdict(list)
    
    # Process each time slot
    for slot in time_slots:
        parsed = parse_time_slot(slot)
        if not parsed:
            continue
            
        # Count available participants
        available_participants = []
        for participant, slots in participants_data.items():
            if slots.get(slot) == 'Yes':
                available_participants.append(participant)
        
        # Extract day for grouping
        day_key = f"{parsed['day_name']} {parsed['date']}"
        
        # Extract time portion for display (handle both 24h and 12h formats)
        # Try 24-hour format first
        time_match = re.search(r'(\d{1,2}:00\s*-\s*\d{1,2}:00)(?!\s*[AP]M)', slot)
        if not time_match:
            # Try 12-hour format with AM/PM
            time_match = re.search(r'(\d{1,2}:00\s*(?:AM|PM)\s*-\s*\d{1,2}:00\s*(?:AM|PM))', slot)
        
        time_only = time_match.group(1) if time_match else slot
        
        slot_data = {
            'time_slot': slot,
            'time_only': time_only,
            'start_hour': parsed['start_hour'],  # 24-hour format for sorting (0-23)
            'count': len(available_participants),
            'participants': available_participants,
            'total_participants': len(participants_data)
        }
        
        slots_by_day[day_key].append(slot_data)
        availability_matrix[slot] = slot_data
    
    # Sort slots within each day by time (using 24-hour format)
    for day in slots_by_day:
        slots_by_day[day].sort(key=lambda x: x['start_hour'])
    
    return availability_matrix, dict(slots_by_day)

def get_availability_color_class(count, total):
    """Get CSS class based on availability percentage"""
    if count == 0:
        return 'available-0'
    elif count == total:
        return 'available-all'
    else:
        # Calculate percentage-based classes
        percentage = (count / total) * 100
        if percentage <= 25:
            return 'available-low'
        elif percentage <= 50:
            return 'available-medium'
        elif percentage <= 75:
            return 'available-high'
        else:
            return 'available-most'

def generate_html(participants_data, availability_matrix, slots_by_day):
    """Generate the complete HTML file"""
    participants = list(participants_data.keys())
    total_participants = len(participants)
    total_slots = sum(len(slots) for slots in slots_by_day.values())
    all_available_count = sum(1 for slot in availability_matrix.values() 
                             if slot['count'] == total_participants)
    
    # Convert data to JSON for JavaScript
    availability_json = json.dumps(slots_by_day, indent=2)
    participants_json = json.dumps(participants)
    
    html = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reading Group Availability Timetable</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Oxygen, Ubuntu, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }}
        
        .container {{
            max-width: 1600px;
            margin: 0 auto;
            background: white;
            border-radius: 20px;
            box-shadow: 0 25px 50px -12px rgba(0, 0, 0, 0.25);
            overflow: hidden;
        }}
        
        .header {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 30px;
            text-align: center;
        }}
        
        h1 {{
            font-size: 2.5rem;
            margin-bottom: 10px;
            font-weight: 700;
        }}
        
        .subtitle {{
            font-size: 1.1rem;
            opacity: 0.95;
        }}
        
        .content {{
            padding: 30px;
        }}
        
        .stats {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin-bottom: 30px;
        }}
        
        .stat-card {{
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            padding: 20px;
            border-radius: 15px;
            text-align: center;
            transition: transform 0.3s ease;
        }}
        
        .stat-card:hover {{
            transform: translateY(-5px);
        }}
        
        .stat-value {{
            font-size: 2.5rem;
            font-weight: bold;
            color: #5a67d8;
            margin-bottom: 5px;
        }}
        
        .stat-label {{
            color: #4a5568;
            font-size: 0.9rem;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }}
        
        .controls {{
            background: #f8f9fa;
            padding: 25px;
            border-radius: 15px;
            margin-bottom: 30px;
            border: 2px solid #e9ecef;
        }}
        
        .control-group {{
            margin-bottom: 20px;
        }}
        
        label {{
            display: block;
            font-weight: 600;
            color: #2d3748;
            margin-bottom: 8px;
            font-size: 0.95rem;
        }}
        
        input[type="text"] {{
            width: 100%;
            padding: 12px 15px;
            border: 2px solid #cbd5e0;
            border-radius: 10px;
            font-size: 15px;
            transition: all 0.3s ease;
        }}
        
        input[type="text"]:focus {{
            outline: none;
            border-color: #667eea;
            box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1);
        }}
        
        .button-group {{
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
        }}
        
        button {{
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            padding: 12px 24px;
            border-radius: 10px;
            cursor: pointer;
            font-size: 15px;
            font-weight: 600;
            transition: all 0.3s ease;
            white-space: nowrap;
        }}
        
        button:hover {{
            transform: translateY(-2px);
            box-shadow: 0 10px 20px rgba(102, 126, 234, 0.3);
        }}
        
        button:active {{
            transform: translateY(0);
        }}
        
        #presenterStatus {{
            margin-top: 15px;
            padding: 12px;
            border-radius: 10px;
            font-weight: 600;
            display: none;
        }}
        
        #presenterStatus.success {{
            background: #d4edda;
            color: #155724;
            border: 1px solid #c3e6cb;
            display: block;
        }}
        
        #presenterStatus.error {{
            background: #f8d7da;
            color: #721c24;
            border: 1px solid #f5c6cb;
            display: block;
        }}
        
        #presenterStatus.warning {{
            background: #fff3cd;
            color: #856404;
            border: 1px solid #ffeeba;
            display: block;
        }}
        
        .legend {{
            display: flex;
            justify-content: center;
            gap: 20px;
            margin: 25px 0;
            flex-wrap: wrap;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 15px;
        }}
        
        .legend-item {{
            display: flex;
            align-items: center;
            gap: 10px;
        }}
        
        .legend-color {{
            width: 30px;
            height: 30px;
            border-radius: 8px;
            border: 2px solid #e2e8f0;
        }}
        
        .table-wrapper {{
            overflow-x: auto;
            border-radius: 15px;
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
            margin-top: 20px;
        }}
        
        table {{
            width: 100%;
            border-collapse: separate;
            border-spacing: 0;
            background: white;
        }}
        
        th, td {{
            padding: 12px;
            text-align: center;
            border-right: 1px solid #e2e8f0;
            border-bottom: 1px solid #e2e8f0;
        }}
        
        th {{
            background: linear-gradient(135deg, #4a5568 0%, #2d3748 100%);
            color: white;
            font-weight: 600;
            position: sticky;
            top: 0;
            z-index: 10;
            font-size: 0.95rem;
        }}
        
        th:first-child {{
            border-top-left-radius: 15px;
        }}
        
        th:last-child {{
            border-top-right-radius: 15px;
            border-right: none;
        }}
        
        td:last-child {{
            border-right: none;
        }}
        
        .day-header {{
            background: linear-gradient(135deg, #5a67d8 0%, #667eea 100%);
            font-size: 1rem;
        }}
        
        .time-cell {{
            background: #f7fafc;
            font-weight: 600;
            text-align: left;
            padding-left: 20px;
            white-space: nowrap;
            position: sticky;
            left: 0;
            z-index: 5;
            border-right: 2px solid #cbd5e0;
        }}
        
        .count-display {{
            font-size: 1.2rem;
            font-weight: 700;
        }}
        
        /* Availability color classes */
        .available-0 {{
            background-color: #fed7d7;
            color: #9b2c2c;
        }}
        
        .available-low {{
            background-color: #feebc8;
            color: #9c4221;
        }}
        
        .available-medium {{
            background-color: #fef5e7;
            color: #975a16;
        }}
        
        .available-high {{
            background-color: #e6fffa;
            color: #234e52;
        }}
        
        .available-most {{
            background-color: #d4f1e9;
            color: #22543d;
        }}
        
        .available-all {{
            background: linear-gradient(135deg, #84fab0 0%, #8fd3f4 100%);
            color: #22543d;
            font-weight: 700;
        }}
        
        .presenter-highlight {{
            background: linear-gradient(135deg, #fa709a 0%, #fee140 100%) !important;
            color: #742a2a !important;
            font-weight: 700;
            animation: pulse 2s infinite;
            position: relative;
        }}
        
        @keyframes pulse {{
            0% {{
                box-shadow: 0 0 0 0 rgba(250, 112, 154, 0.7);
            }}
            70% {{
                box-shadow: 0 0 0 10px rgba(250, 112, 154, 0);
            }}
            100% {{
                box-shadow: 0 0 0 0 rgba(250, 112, 154, 0);
            }}
        }}
        
        .empty-cell {{
            background: #f7fafc;
            color: #a0aec0;
        }}
        
        small {{
            display: block;
            margin-top: 5px;
            color: #718096;
            font-size: 0.85rem;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>📅 Reading Group Availability Timetable</h1>
            <div class="subtitle">Tokyo Time (JST)</div>
        </div>
        
        <div class="content">
            <div class="stats">
                <div class="stat-card">
                    <div class="stat-value">{total_participants}</div>
                    <div class="stat-label">Total Participants</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{total_slots}</div>
                    <div class="stat-label">Time Slots</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{all_available_count}</div>
                    <div class="stat-label">All Available</div>
                </div>
            </div>

            <div class="controls">
                <div class="control-group">
                    <label for="presenterInput">
                        🎯 Enter Names to Check Availability (comma-separated):
                    </label>
                    <input type="text" 
                           id="presenterInput" 
                           placeholder="e.g., {', '.join(participants[:min(3, len(participants))])}"
                           autocomplete="off">
                    <small>Total participants: {total_participants} | Enter any number of names to find common availability</small>
                </div>
                <div class="button-group">
                    <button onclick="highlightPresenters()">🔍 Find Common Availability</button>
                    <button onclick="clearHighlight()">🔄 Clear Highlights</button>
                    <button onclick="showAllAvailable()">✅ Show Fully Available</button>
                    <button onclick="exportResults()">📊 Export Results</button>
                </div>
                <div id="presenterStatus"></div>
            </div>

            <div class="legend">
                <div class="legend-item">
                    <div class="legend-color available-0"></div>
                    <span>None Available</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color available-low"></div>
                    <span>1-25% Available</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color available-medium"></div>
                    <span>26-50% Available</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color available-high"></div>
                    <span>51-75% Available</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color available-most"></div>
                    <span>76-99% Available</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color available-all"></div>
                    <span>All Available</span>
                </div>
                <div class="legend-item">
                    <div class="legend-color presenter-highlight"></div>
                    <span>Selected Match</span>
                </div>
            </div>

            <div class="table-wrapper">
                <table id="availabilityTable">
                    <!-- Table will be generated by JavaScript -->
                </table>
            </div>
        </div>
    </div>

    <script>
        // Data from Python
        const availabilityData = {availability_json};
        const participants = {participants_json};
        const totalParticipants = participants.length;
        
        // Build the table
        function buildTable() {{
            const table = document.getElementById('availabilityTable');
            table.innerHTML = '';
            
            // Get all days and sort them
            const days = Object.keys(availabilityData).sort((a, b) => {{
                const dayOrder = {{'Sun': 0, 'Mon': 1, 'Tue': 2, 'Wed': 3, 'Thu': 4, 'Fri': 5, 'Sat': 6}};
                const dayA = a.split(' ')[0];
                const dayB = b.split(' ')[0];
                const dateA = parseInt(a.split(' ')[1]);
                const dateB = parseInt(b.split(' ')[1]);
                
                if (dayOrder[dayA] !== dayOrder[dayB]) {{
                    return dayOrder[dayA] - dayOrder[dayB];
                }}
                return dateA - dateB;
            }});
            
            // Create header row
            const headerRow = document.createElement('tr');
            const timeHeader = document.createElement('th');
            timeHeader.textContent = 'Time Slot';
            timeHeader.className = 'time-header';
            headerRow.appendChild(timeHeader);
            
            days.forEach(day => {{
                const dayHeader = document.createElement('th');
                dayHeader.textContent = day + ' Aug';
                dayHeader.className = 'day-header';
                headerRow.appendChild(dayHeader);
            }});
            table.appendChild(headerRow);
            
            // Get all unique time slots with proper sorting
            const timeSlotMap = new Map();
            
            days.forEach(day => {{
                availabilityData[day].forEach(slot => {{
                    if (!timeSlotMap.has(slot.time_only)) {{
                        // Parse the time for sorting - handle both 24h and 12h formats
                        let hour24;
                        
                        // Try 24-hour format first (e.g., "14:00 - 15:00")
                        const match24h = slot.time_only.match(/(\d{{1,2}}):00\s*-\s*\d{{1,2}}:00/);
                        if (match24h && !slot.time_only.includes('AM') && !slot.time_only.includes('PM')) {{
                            hour24 = parseInt(match24h[1], 10);
                        }} else {{
                            // Try 12-hour format (e.g., "2:00 PM - 3:00 PM")
                            const match12h = slot.time_only.match(/(\d{{1,2}}):00\s*(AM|PM)/);
                            if (match12h) {{
                                let hour = parseInt(match12h[1], 10);
                                const period = match12h[2];
                                
                                if (period === 'AM') {{
                                    hour24 = (hour === 12) ? 0 : hour;
                                }} else {{
                                    hour24 = (hour === 12) ? 12 : hour + 12;
                                }}
                            }} else {{
                                hour24 = 0; // Default if parsing fails
                            }}
                        }}
                        
                        timeSlotMap.set(slot.time_only, hour24);
                    }}
                }});
            }});
            
            // Sort time slots by their 24-hour value
            const sortedTimeSlots = Array.from(timeSlotMap.entries())
                .sort((a, b) => a[1] - b[1])
                .map(entry => entry[0]);
            
            // Log for debugging
            console.log('Time slots in chronological order:');
            sortedTimeSlots.forEach((slot, idx) => {{
                console.log(`${{idx + 1}}. ${{slot}} (Hour: ${{timeSlotMap.get(slot)}})`);
            }});
            
            // Create rows for each time slot
            sortedTimeSlots.forEach(timeSlot => {{
                const row = document.createElement('tr');
                
                // Time cell
                const timeCell = document.createElement('td');
                timeCell.textContent = timeSlot;
                timeCell.className = 'time-cell';
                row.appendChild(timeCell);
                
                // Data cells for each day
                days.forEach(day => {{
                    const cell = document.createElement('td');
                    const slotData = availabilityData[day].find(s => s.time_only === timeSlot);
                    
                    if (slotData) {{
                        const percentage = (slotData.count / totalParticipants) * 100;
                        let className = 'available-0';
                        
                        if (slotData.count === 0) {{
                            className = 'available-0';
                        }} else if (slotData.count === totalParticipants) {{
                            className = 'available-all';
                        }} else if (percentage <= 25) {{
                            className = 'available-low';
                        }} else if (percentage <= 50) {{
                            className = 'available-medium';
                        }} else if (percentage <= 75) {{
                            className = 'available-high';
                        }} else {{
                            className = 'available-most';
                        }}
                        
                        cell.className = className;
                        cell.innerHTML = `<div class="count-display">${{slotData.count}}/${{totalParticipants}}</div>`;
                        cell.dataset.participants = JSON.stringify(slotData.participants);
                        cell.dataset.count = slotData.count;
                        cell.dataset.timeSlot = `${{day}} Aug ${{timeSlot}}`;
                    }} else {{
                        cell.className = 'empty-cell';
                        cell.textContent = '-';
                    }}
                    
                    row.appendChild(cell);
                }});
                
                table.appendChild(row);
            }});
        }}
        
        function highlightPresenters() {{
            const input = document.getElementById('presenterInput').value;
            const status = document.getElementById('presenterStatus');
            
            if (!input.trim()) {{
                status.className = 'warning';
                status.textContent = '⚠️ Please enter at least one name to check availability';
                return;
            }}
            
            // Parse input names
            const requestedNames = input.split(',').map(p => p.trim()).filter(p => p);
            const requestedNamesLower = requestedNames.map(n => n.toLowerCase());
            
            // Check which names are valid participants
            const participantsLower = participants.map(p => p.toLowerCase());
            const validNames = [];
            const invalidNames = [];
            
            requestedNames.forEach((name, idx) => {{
                const lowerName = requestedNamesLower[idx];
                const matchIndex = participantsLower.indexOf(lowerName);
                if (matchIndex !== -1) {{
                    validNames.push(participants[matchIndex]);
                }} else {{
                    invalidNames.push(name);
                }}
            }});
            
            if (invalidNames.length > 0) {{
                status.className = 'error';
                status.textContent = `❌ Unknown participant(s): ${{invalidNames.join(', ')}}. Valid participants: ${{participants.join(', ')}}`;
                return;
            }}
            
            if (validNames.length === 0) {{
                status.className = 'error';
                status.textContent = '❌ No valid participants found';
                return;
            }}
            
            // Clear previous highlights
            clearHighlight();
            
            // Find matching slots
            let foundCount = 0;
            const matchingSlots = [];
            const cells = document.querySelectorAll('td[data-participants]');
            
            cells.forEach(cell => {{
                const cellParticipants = JSON.parse(cell.dataset.participants).map(p => p.toLowerCase());
                const allRequestedAvailable = validNames.every(name => 
                    cellParticipants.includes(name.toLowerCase())
                );
                
                if (allRequestedAvailable) {{
                    cell.classList.add('presenter-highlight');
                    foundCount++;
                    if (cell.dataset.timeSlot) {{
                        matchingSlots.push(cell.dataset.timeSlot);
                    }}
                }}
            }});
            
            // Update status message
            if (foundCount > 0) {{
                status.className = 'success';
                const namesList = validNames.length === 1 ? validNames[0] : 
                    validNames.length === 2 ? `${{validNames[0]}} and ${{validNames[1]}}` :
                    `all ${{validNames.length}} selected participants`;
                status.innerHTML = `✅ Found <strong>${{foundCount}}</strong> time slot(s) where ${{namesList}} are available`;
            }} else {{
                status.className = 'error';
                status.textContent = `❌ No time slots found where all selected participants are simultaneously available`;
            }}
        }}
        
        function clearHighlight() {{
            document.querySelectorAll('.presenter-highlight').forEach(cell => {{
                cell.classList.remove('presenter-highlight');
            }});
            document.getElementById('presenterStatus').className = '';
            document.getElementById('presenterStatus').textContent = '';
        }}
        
        function showAllAvailable() {{
            clearHighlight();
            let count = 0;
            const matchingSlots = [];
            
            const cells = document.querySelectorAll('td[data-count]');
            cells.forEach(cell => {{
                if (parseInt(cell.dataset.count) === totalParticipants) {{
                    cell.classList.add('presenter-highlight');
                    count++;
                    if (cell.dataset.timeSlot) {{
                        matchingSlots.push(cell.dataset.timeSlot);
                    }}
                }}
            }});
            
            const status = document.getElementById('presenterStatus');
            if (count > 0) {{
                status.className = 'success';
                status.innerHTML = `✅ Highlighting <strong>${{count}}</strong> time slot(s) where all <strong>${{totalParticipants}}</strong> participants are available`;
            }} else {{
                status.className = 'warning';
                status.textContent = '⚠️ No time slots found where all participants are available';
            }}
        }}
        
        function exportResults() {{
            const highlighted = document.querySelectorAll('.presenter-highlight');
            if (highlighted.length === 0) {{
                alert('Please select participants or show all available slots first');
                return;
            }}
            
            let csvContent = 'Time Slot,Available Count,Participants\\n';
            highlighted.forEach(cell => {{
                if (cell.dataset.timeSlot && cell.dataset.participants) {{
                    const participants = JSON.parse(cell.dataset.participants).join('; ');
                    csvContent += `"${{cell.dataset.timeSlot}}","${{cell.dataset.count}}/${{totalParticipants}}","${{participants}}"\\n`;
                }}
            }});
            
            // Create download link
            const blob = new Blob([csvContent], {{ type: 'text/csv' }});
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement('a');
            a.href = url;
            a.download = 'availability_results.csv';
            a.click();
            window.URL.revokeObjectURL(url);
        }}
        
        // Build table on load
        buildTable();
        
        // Add enter key support for input
        document.getElementById('presenterInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                highlightPresenters();
            }}
        }});
    </script>
</body>
</html>'''
    
    return html

def main():
    """Main function to generate the timetable"""
    # Configuration
    csv_filepath = 'Tv=v_Reading_Group-202508171456.csv'  # Updated filename
    output_filepath = 'reading_group_timetable.html'
    
    print("="*60)
    print("READING GROUP AVAILABILITY TIMETABLE GENERATOR")
    print("="*60)
    
    print("\n📂 Reading CSV file...")
    participants_data, time_slots = read_csv_file(csv_filepath)
    
    print(f"\n👥 Found {len(participants_data)} participants:")
    for i, participant in enumerate(participants_data.keys(), 1):
        print(f"   {i}. {participant}")
    
    print(f"\n⏰ Processing {len(time_slots)} time slots...")
    
    # Show sample time slots for debugging
    if time_slots:
        print(f"   Sample time slots:")
        for slot in time_slots[:3]:
            print(f"     - {slot}")
    
    availability_matrix, slots_by_day = process_availability_data(
        participants_data, time_slots
    )
    
    print(f"\n📅 Days covered (in chronological order):")
    if slots_by_day:
        day_order = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
        sorted_days = sorted(slots_by_day.keys(), key=lambda d: (day_order.index(d.split()[0]), int(d.split()[1])))
        
        for day in sorted_days:
            slots_count = len(slots_by_day[day])
            print(f"   • {day} Aug: {slots_count} time slots")
        
        # Show time slot order for verification
        if sorted_days:
            first_day = sorted_days[0]
            print(f"\n🕐 Time slot order for {first_day} (earliest to latest):")
            for slot in slots_by_day[first_day]:
                time_str = slot['time_only']
                hour_24 = slot['start_hour']
                print(f"   {time_str:20s} (24-hour value: {hour_24:02d})")
        
        # Update slots_by_day to use sorted day order
        slots_by_day = {day: slots_by_day[day] for day in sorted_days}
    else:
        print("   ⚠️ No valid days found")
        sorted_days = []
    
    # Statistics
    total_slots = sum(len(slots) for slots in slots_by_day.values())
    all_available = sum(1 for slot in availability_matrix.values() 
                       if slot['count'] == len(participants_data))
    
    print(f"\n📊 Statistics:")
    print(f"   • Total time slots: {total_slots}")
    print(f"   • Slots where ALL participants available: {all_available}")
    if total_slots > 0:
        print(f"   • Percentage fully available: {(all_available/total_slots)*100:.1f}%")
    else:
        print(f"   • Percentage fully available: N/A (no valid time slots)")
    
    # Find best time slots
    if availability_matrix:
        print(f"\n🌟 Best availability (top 5 slots):")
        sorted_slots = sorted(availability_matrix.values(), 
                             key=lambda x: x['count'], 
                             reverse=True)[:5]
        for i, slot in enumerate(sorted_slots, 1):
            slot_display = slot['time_slot'][:50] if len(slot['time_slot']) > 50 else slot['time_slot']
            print(f"   {i}. {slot_display} - {slot['count']}/{len(participants_data)} available")
    else:
        print(f"\n⚠️ No valid time slots found in the CSV file")
    
    print(f"\n💾 Generating HTML...")
    html_content = generate_html(participants_data, availability_matrix, slots_by_day)
    
    print(f"📝 Writing to {output_filepath}...")
    with open(output_filepath, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print("\n" + "="*60)
    print(f"✅ SUCCESS! Generated {output_filepath}")
    print("="*60)
    
    if total_slots > 0:
        print(f"\n🌐 Open the file in a web browser to view the interactive timetable")
        print("💡 Tips:")
        print("   • Enter any number of names to find common availability")
        print("   • Use the export button to save results as CSV")
        print("   • The table shows availability counts, not individual names")
    else:
        print(f"\n⚠️ Warning: No valid time slots were found in the CSV file.")
        print("   Please check that your CSV file has the correct format:")
        print("   • Column headers should contain 'Aug 2025'")
        print("   • Time format should be like 'Mon 18 Aug 2025 09:00 - 10:00'")
        print("   • Or '24-hour format: Mon 18 Aug 2025 09:00 - 10:00'")
        print("   • Or '12-hour format: Mon 18 Aug 2025 9:00 AM - 10:00 AM'")

if __name__ == "__main__":
    main()

READING GROUP AVAILABILITY TIMETABLE GENERATOR

📂 Reading CSV file...

👥 Found 5 participants:
   1. Shu
   2. Nisha
   3. Longye
   4. Jingni
   5. Humphrey

⏰ Processing 50 time slots...
   Sample time slots:
     - Mon 18 Aug 2025 08:00 - 09:00
     - Mon 18 Aug 2025 09:00 - 10:00
     - Mon 18 Aug 2025 10:00 - 11:00

📅 Days covered (in chronological order):
   • Mon 18 Aug: 10 time slots
   • Tue 19 Aug: 10 time slots
   • Wed 20 Aug: 10 time slots
   • Thu 21 Aug: 10 time slots
   • Fri 22 Aug: 10 time slots

🕐 Time slot order for Mon 18 (earliest to latest):
   08:00 - 09:00        (24-hour value: 08)
   09:00 - 10:00        (24-hour value: 09)
   10:00 - 11:00        (24-hour value: 10)
   11:00 - 12:00        (24-hour value: 11)
   12:00 - 13:00        (24-hour value: 12)
   13:00 - 14:00        (24-hour value: 13)
   14:00 - 15:00        (24-hour value: 14)
   15:00 - 16:00        (24-hour value: 15)
   16:00 - 17:00        (24-hour value: 16)
   17:00 - 18:00        (24-hour 

<>:867: SyntaxWarning: invalid escape sequence '\d'
<>:867: SyntaxWarning: invalid escape sequence '\s'
<>:867: SyntaxWarning: invalid escape sequence '\d'
<>:867: SyntaxWarning: invalid escape sequence '\s'
<>:867: SyntaxWarning: invalid escape sequence '\d'
<>:867: SyntaxWarning: invalid escape sequence '\s'
<>:867: SyntaxWarning: invalid escape sequence '\d'
<>:867: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_169112/916593926.py:867: SyntaxWarning: invalid escape sequence '\d'
  </html>'''
/tmp/ipykernel_169112/916593926.py:867: SyntaxWarning: invalid escape sequence '\s'
  </html>'''
/tmp/ipykernel_169112/916593926.py:867: SyntaxWarning: invalid escape sequence '\d'
  </html>'''
/tmp/ipykernel_169112/916593926.py:867: SyntaxWarning: invalid escape sequence '\s'
  </html>'''
